In [ ]:
from setup import setup,scale_features
from utils import find_best_hdbscan_params, apply_pca
import hdbscan
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import silhouette_score

In [ ]:
features = setup()
scaled_features = scale_features(features)

In [ ]:
best_size, best_samples = find_best_hdbscan_params(scaled_features)

clusterer = hdbscan.HDBSCAN(min_cluster_size=best_size, min_samples=best_samples)
regime = clusterer.fit_predict(scaled_features)

features_hdbscan = pd.DataFrame(
	scaled_features,
	index=features.index,
	columns=features.columns
)
features_hdbscan['regime'] = regime

In [ ]:
#calculate silhouette score
mask_noise = regime != -1
features_hdbscan_filtered = features_hdbscan[mask_noise]
num_clusters = len(set(regime)) - (1 if -1 in regime else 0)

print(num_clusters)

if num_clusters >= 2:
    score = silhouette_score(scaled_features[mask_noise], regime[mask_noise])
    print(f'Silhouette Score: {score}')


In [ ]:
plt.figure(figsize=(12, 6))
plt.scatter(features.index, features_hdbscan['avg_return'].loc[features.index], c=regime, cmap='viridis', s=10)
plt.title("Avg Return with HDBSCAN Regimes")
plt.xlabel("Date")
plt.ylabel("Price")
plt.colorbar(label="Regime")
plt.show()

In [ ]:
# Run hdbscan on PCA features
regime, pca = apply_pca(clusterer, scaled_features, 5)

features_hdbscan_pca = pd.DataFrame(
    pca,
    columns=[f'PC{i+1}' for i in range(pca.shape[1])]
)
features_hdbscan_pca['regime'] = regime

print(features_hdbscan_pca.head())

plt.figure(figsize=(12, 6))
plt.scatter(features_hdbscan_pca['PC1'], features_hdbscan_pca['PC2'], c=regime, cmap='viridis', s=10)
plt.title("HDBSCAN Clusters in PCA Space")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.colorbar(label="Regime")
plt.show()

In [ ]:
features_hdbscan_pca['avg_return'] = features['avg_return']
features_hdbscan_pca.groupby('regime')['avg_return'].mean()

plt.figure(figsize=(12, 6))
plt.scatter(features_hdbscan_pca['PC1'], features_hdbscan_pca['PC2'], c=features['avg_return'], cmap='viridis', s=10)
plt.title("HDBSCAN Clusters Colored by Avg Return")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.colorbar(label="Avg Return")
plt.show()

In [ ]:
features_hdbscan.assign(regime=regime).groupby('regime').mean()

In [ ]:
features_hdbscan["future_return"] = features_hdbscan["avg_return"].shift(-1)
features_hdbscan.assign(regime=regime).groupby('regime')["future_return"].mean()